# From Model to Field
## Engineering Reliable Edge AI Applications on Embedded GPUs

Companion notebook for the talk at [Tokyo AI (TAI)](https://luma.com/xkb4ai5f).

The running example is **AIPV1EMI0001**, a YOLOv7-tiny `car` / `person` / `truck` detector packaged
as an EdgeAI Station *AI Processor* and published with its build notebook
([`localdev.ipynb`](localdev.ipynb)), weights on
[Hugging Face](https://huggingface.co/Edgematrix-JP/AIPV1EMI0001) and dashboard.
Every number shown here comes from a run recorded in this repository. This notebook explains the
engineering decisions; [`localdev.ipynb`](localdev.ipynb) is the hands-on rebuild. Neither notebook
trains a model from scratch.

There is no code to run here: every command, with its checks, is in
[`localdev.ipynb`](localdev.ipynb).

## Agenda

1. A model is not an application
2. The AI Processor: one contract from model to dashboard
3. The GPU video pipeline
4. From ONNX to a TensorRT engine
5. Measuring fairly
6. Packaging reproducibly
7. Validating in layers
8. Operating in the field
9. Checklist

## 1. A model is not an application

A notebook demo answers *“does the model detect cars?”*. A field deployment must answer much more:

| In the lab | In the field |
|---|---|
| One video file, one GPU, one user | Up to 8 cameras sharing one embedded GPU and its memory with other services |
| Python loop around `model(frame)` | Hardware decode, batching, tracking, analytics, 24/7 |
| Latest framework from `pip` | Pinned SDK stack (JetPack, TensorRT, DeepStream) per device generation |
| “It ran once” | Repeatable build, verified package, acceptance test, versioned output contract |
| Results on screen | Business data delivered to applications and dashboards |

The rest of the talk is the engineering that closes that gap.

## 2. The AI Processor: one contract from model to dashboard

An **AI Processor (AIP)** is a self-describing folder. The station assembles the video pipeline
from the manifest and configuration files; the author supplies these declarative settings and
the application components rather than writing the pipeline orchestrator.

```mermaid
flowchart LR
    subgraph AIP["AI Processor"]
        M["manifest.json<br/>what it is"]
        C["configs/<br/>how to infer and track"]
        MO["models/<br/>engine, labels, parser"]
        CB["callback.py<br/>what it outputs"]
        L["logic/ + dashboard/<br/>how it is used"]
    end
    AIP -- "eais-tools aip build / export" --> PKG[["verified .tgz"]]
    PKG --> ST["EdgeAI Station"]
    ST --> APP["Applications<br/>Node-RED, dashboards"]
```

The manifest describes the AIP to the station; model and tracker behavior is configured in the
referenced files:

**AIPV1EMI0001 at a glance** (from `manifest.json`):

| Field | Value |
|---|---|
| Identity | `AIPV1EMI0001` version **6.1.0** |
| Hardware targets | EX3, EX5, EX7 |
| Input streams | up to 8 |
| Detector | Yolov7, classes `car`, `person`, `truck`, batch 8 |
| Tracker | NvDCF, 640×384 |
| Analytics | ROI, LCC (region of interest, line crossing) |

## 3. The GPU video pipeline

```mermaid
flowchart LR
    SRC["Cameras<br/>RTSP / files"] --> DEC["Hardware decode"]
    DEC --> MUX["Batch frames<br/>from all streams"]
    MUX --> INF["Inference<br/>TensorRT engine"]
    INF --> TRK["Tracking<br/>NvDCF"]
    TRK --> ANA["Analytics<br/>ROI, line crossing"]
    ANA --> CB["callback.py"]
    CB --> NR["Node-RED"]
    NR --> UI["Dashboard"]
```

Design principles:

- **Keep frames in GPU-accessible buffers** where the DeepStream path supports it; avoid
  unnecessary CPU copies. CPU cores still handle orchestration, callbacks and other services.
- **Batch across cameras**: a batch lets inference process frames from multiple streams together;
  the throughput benefit depends on workload and batch size.
- **In-process inference server** (`nvinferserver`): no network hop between video and model.
- **Track, then count**: associating detections across frames supports more stable counts and
  line-crossing analytics; results still depend on tracker configuration and scene conditions.
- **Callbacks consume metadata, not image tensors**: `callback.py` aggregates per-second counts;
  encoded HLS or WebRTC video follows a separate path to the dashboard.

## 4. From ONNX to a TensorRT engine

```mermaid
flowchart LR
    T["Training<br/>PyTorch"] --> O["ONNX<br/>more portable; includes TRT NMS plugin"]
    O -- "trtexec on the target device" --> E["TensorRT engine<br/>FP16, batch 1 to 8"]
    O -. "published" .-> HF[("Hugging Face")]
    E -. "published, tagged<br/>jp6-l4t36.3-arm64" .-> HF
```

| Artifact | Portability | Role |
|---|---|---|
| ONNX | More portable than an engine, but this graph includes `EfficientNMS_TRT`; its plugin and operator support must match the target stack | Source for rebuilding and validation |
| TensorRT engine | **No**: tied to the TensorRT version, GPU architecture and build profile | What actually runs; shipped prebuilt for one tested stack |

Every optimization is a claim. Pair each one with a **measurement** (did it help, and by how
much?) and a **check** (does the result still work correctly?); treat any claim without both as
an assumption.

```mermaid
flowchart LR
    O["Optimize<br/>build choice"] --> M["Measure<br/>same method for every variant"]
    M --> V["Validate<br/>still loads, runs and detects"]
    V -- "slower, or wrong output" --> O
```

Engineering decisions for this AIP:

| Optimization | Measured by | Validated by |
|---|---|---|
| **TensorRT engine** built on the target | TensorRT FP32 vs ONNX Runtime CUDA FP32: 1.3–1.5× | Engine smoke test on the device |
| **FP16** | TensorRT FP16 vs TensorRT FP32: 3.6–4.0× | Random-output differences are sanity only; compare labelled detections in [`accuracy.ipynb`](accuracy.ipynb) |
| **Dynamic batch 1 / 8 / 8**: one engine for 1 to 8 streams | Latency at batch 1, 4 and 8 | Config validators; same batch in engine, Triton, DeepStream and manifest |
| **NMS in the graph** (`EfficientNMS_TRT`) + C++ parser | Contributes to pipeline throughput; not isolated by the raw benchmark | Pipeline test requires detections |
| **Workspace 1024 MiB** (shared CPU/GPU memory) | A 4096 MiB build was OOM-killed on a running station | Build completes; smoke test |

The 4.8–5.6× of TensorRT FP16 over ONNX Runtime CUDA FP32 combines precision and runtime
effects, including ONNX Runtime's CPU nodes: report the factors separately.

## 5. Measuring fairly

*“TensorRT is N× faster than ONNX”* is meaningless unless both sides are measured the same way.
Common traps:

- `trtexec` reports GPU compute time and **pipelines** requests; a Python ONNX Runtime loop
  includes dispatch and synchronization. Different boundaries, different numbers.
- ONNX Runtime's TensorRT provider builds **its own** engine: it does not measure yours.
- TF32 enabled on one side only; different batch, input copies inside the timer, cold start.

The method used here (following [Ultralytics' predictor](https://github.com/ultralytics/ultralytics/blob/80c34c050921abd29970d8e3d2a2a41ec6538c98/ultralytics/engine/predictor.py#L380-L397)):

| Control | Method |
|---|---|
| Graph | One NMS-free copy of the model |
| Precision flags | ONNX Runtime CUDA uses `use_tf32=0`; TensorRT FP32 uses `--noTF32`; TensorRT FP16 adds `--fp16` |
| Data | One GPU-resident input per batch; preallocated GPU outputs |
| Timer | `cudaDeviceSynchronize → perf_counter → one call → cudaDeviceSynchronize` |
| Statistics | 20 warm-up + 50 timed requests × 3 rounds, shuffled order, median and p95 |
| Placement | Profile untimed ONNX Runtime requests on the same session, then stop profiling |
| Conditions | Versions, file hashes and GPU load/clock/temperature around each round |

Two measurements answer two different questions; do not mix them:

- **Raw model latency**: milliseconds per inference request, one at a time, without NMS.
  Answers *“how fast does this runtime execute the model on this station?”*
- **Pipeline throughput** (`eais-tools check perf`): frames per second across all streams,
  including decode, tracking and callback. Answers *“how many cameras can this station handle?”*

Implementation: [`tools/benchmark.py`](tools/benchmark.py), run from section 3 of
[`localdev.ipynb`](localdev.ipynb).

Reference run: Jetson Orin NX 16 GB, JetPack 6.0 (L4T R36.3), TensorRT 8.6.2,
ONNX Runtime 1.19.0. Median latency per request, with the speed-up over
ONNX Runtime CUDA FP32 at the same batch size
([raw results](doc/benchmarks/orin-nx-16gb-jp6.0.json)):

| Variant | Batch 1 | Batch 4 | Batch 8 |
|---|---:|---:|---:|
| ONNX Runtime CUDA FP32 | 22.3 ms (1.00×) | 81.5 ms (1.00×) | 139.2 ms (1.00×) |
| TensorRT FP32 | 16.6 ms (1.34×) | 56.1 ms (1.45×) | 108.0 ms (1.29×) |
| TensorRT FP16 | 4.7 ms (4.76×) | 14.6 ms (5.58×) | 27.1 ms (5.13×) |

ONNX Runtime profiled nodes: 173 CUDA, 9 CPU.

![Median latency and images per second by batch size](doc/benchmarks/orin-nx-16gb-jp6.0.png)

### Reading the results

- **TensorRT FP16 is 4.8–5.6× faster than ONNX Runtime CUDA FP32** on the same graph and timer;
  TensorRT FP32 alone gives 1.3–1.5× over ONNX Runtime CUDA FP32, and TensorRT FP16 is
  3.6–4.0× faster than TensorRT FP32. The larger gap also reflects the backend/runtime difference.
- Video pipelines were stopped; other station services remained running. GPU load was 0% in ten
  samples before the run, and the GPU clock was 918 MHz at the round snapshots.
- The ONNX Runtime profile observed **173 CUDA nodes and 9 CPU nodes**; the latter were integer
  shape operations. The timing includes both providers, so this is not a GPU-only comparison.
- FP16 outputs differ from FP32 (some raw box coordinates by up to 14 px on random
  input): a numerical sanity check, **not** an accuracy evaluation. The companion
  [`accuracy.ipynb`](accuracy.ipynb) evaluates labelled detections separately.
- This is raw model latency. Pipeline throughput (decode + batch + track + analytics) is
  measured separately, with `eais-tools check perf`.

## Labelled accuracy: parity is not quality

On all 5,000 COCO val2017 images, with the same raw-model postprocessing and batch 1:

| Runtime | mAP 50:95 | AP 50 |
|---|---:|---:|
| ONNX Runtime FP32 | 12.64% | 23.28% |
| TensorRT FP32 | 12.64% | 23.28% |
| TensorRT FP16 | 12.64% | 23.26% |

FP16 passed the **provisional** FP32 parity tolerances at batches 1, 4 and 8. Yet the direct
deployment engine's recall at confidence 0.35 / IoU 0.50 was **41.66% car, 23.96% person
and 15.22% truck**. A relative parity pass does not make these absolute results acceptable.

The engine filters low scores, so standard AP is not reported for it. Direct engine results
do not include DeepStream parsing or tracking; training-set overlap and target-scene quality
are unknown. Inspect misses and validate on labelled target-camera data.
[Method and full aggregate results](doc/benchmarks/orin-nx-16gb-jp6.0-accuracy.json).

## 6. Packaging reproducibly

```mermaid
flowchart LR
    SRC["Repository<br/>+ built engine and parser"] --> B["eais-tools aip build<br/>--include-file .include"]
    B --> D["dist/"]
    D --> X["eais-tools aip export"]
    X --> A[["AIPV1EMI0001_dist.tgz"]]
    A -. "release CI" .-> CK["Additional file-set<br/>and archive integrity checks"]
```

- **Native packaging**: EAIS Tools builds and exports the runtime package. `.include` adds
  files beyond its defaults; no second builder is needed in the tutorial.
- **Explicit additions**: this repository's `.include` names individual files. Directories would also
  ship `node_modules/`, caches or `flows_cred.json` (credentials).
- **Release safeguards in CI**: additional checks reject unexpected files, changed bytes and
  unsafe archive entries. They are not extra steps in the developer's build notebook.
- **Weights outside Git**, versioned on Hugging Face and verified by SHA-256 at download.

## 7. Validating in layers

```mermaid
flowchart LR
    S["Static<br/>eais-tools aip check<br/>Triton / DeepStream validators"] --> E["Engine smoke test<br/>loads and runs on this device"]
    E --> L["Labelled model / engine<br/>AP + per-class precision / recall"]
    E --> P["Pipeline test<br/>eais-tools pipeline test<br/>decode, infer, track, callback"]
    P --> C["Output contract<br/>schema + detections > 0"]
    C --> U["Dashboard<br/>human check"]
```

- Static checks are fast but cannot run the tracker or the GPU.
- The **station-managed pipeline test** runs the real pipeline headless, on a sample video.
- **A started pipeline, or a timeout, is not a pass**: the test requires valid callback
  messages *with* detections.
- **Labelled accuracy** is a different check from latency or a pipeline smoke test. Compare
  raw FP32/FP16 on the same images and score the deployment engine at its 0.35 confidence
  threshold with [`accuracy.ipynb`](accuracy.ipynb). The direct engine run does not exercise
  DeepStream parsing, tracking or the callback.
- The callback output is a documented, versioned **contract** (`doc/USER_MANUAL.md`), checked
  automatically: applications depend on it, not on the model.
- **Know what is not proven yet**: detection accuracy on the target scenes (labelled images),
  capacity in streams at the target FPS (`eais-tools check perf`) and 24/7 stability (soak test).

## 8. Operating in the field

- **Business logic as a Node-RED flow** shipped with the AIP (`logic/`): subscribes to the
  pipeline's callback, normalizes counts per class, ROI and line crossing.
- **Dashboard shipped with the AIP** (`dashboard/`, Vue 3 UIBuilder): pipeline-selected HLS or WebRTC video and counts;
  browser libraries are vendored so it works **offline**.
- **No secrets in packages**: `flows_cred.json` never ships; credentials are configured on the station.
- **One package, many stations**: install the verified `.tgz`; rebuild the engine from ONNX
  when the SDK generation changes.
- **Watch the device, not just the model**: memory headroom (shared CPU/GPU), GPU load and
  temperature, stream health.

## 9. Checklist: from model to field

- [ ] Export a portable model (ONNX) and keep it as the source of truth, with checksums
- [ ] Build the engine **on** the target SDK stack; tag prebuilt engines with that stack
- [ ] Choose precision and batch profile deliberately, and size builder memory for a shared device
- [ ] Benchmark with **one** timing boundary for every backend; report conditions and medians
- [ ] Evaluate labelled detection accuracy and FP16 parity, then check target-camera data
- [ ] Package with an allowlist and verify the exact package contents
- [ ] Validate statically, then on the real pipeline, then the output contract
- [ ] Ship the application logic and dashboard with the model, without secrets
- [ ] Make the whole path reproducible by someone else: a notebook, a CI workflow

## Try it yourself

[`localdev.ipynb`](localdev.ipynb) runs every step of this talk on a station, one command per
cell: download and verify the weights, build the parser and engine, benchmark (optional),
validate, package, run the pipeline test and install the dashboard with
`bin/install-nodered-project.sh`. [`accuracy.ipynb`](accuracy.ipynb) evaluates labelled detections.

## Resources

- Source, build notebook and dashboard: [github.com/edge-ai/AIPV1EMI0001](https://github.com/edge-ai/AIPV1EMI0001)
- Weights (ONNX and prebuilt engine): [huggingface.co/Edgematrix-JP/AIPV1EMI0001](https://huggingface.co/Edgematrix-JP/AIPV1EMI0001)
- Benchmark harness and raw results: [`tools/benchmark.py`](tools/benchmark.py),
  [`doc/benchmarks/`](doc/benchmarks/)
- Labelled accuracy tutorial and evaluator: [`accuracy.ipynb`](accuracy.ipynb),
  [`tools/accuracy.py`](tools/accuracy.py), and the
  [COCO reference results](doc/benchmarks/orin-nx-16gb-jp6.0-accuracy.json)
- YOLOv7: [WongKinYiu/yolov7](https://github.com/WongKinYiu/yolov7)
- Talk: [luma.com/xkb4ai5f](https://luma.com/xkb4ai5f)